# Moce zainstalowane: baza bloków vs ENTSO-E

Porównuję moce zainstalowane z pliku `data/power_units_PL_DE_CZ_SK_LT_SE.csv` z mocami zainstalowanymi z ENTSO-E Transparency Platform
(Installed Capacity per Production Type, rok 2025) dla PL, DE, CZ, SK, LT i SE.

Kroki:
1. Wczytanie bazy bloków i zsumowanie mocy po kraju i grupie paliw
2. Pobranie danych z ENTSO-E (przez bibliotekę `entsoe-py`)
3. Porównanie: tabela kraj × paliwo, różnica w MW i %
4. Wykresy słupkowe per kraj

Klucz API jest w pliku `.env` w głównym folderze projektu (linia `ENTSOE_API_KEY=...`), nigdy w kodzie.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import os
from entsoe import EntsoePandasClient

# Wczytanie danych

Wczytuję bazę bloków (492 jednostki). Ścieżka zaczyna się od `../`, bo notatnik leży w folderze `notebooks/`.

In [2]:
power_units = pd.read_csv("../data/power_units_PL_DE_CZ_SK_LT_SE.csv")

In [3]:
power_units.head()

,unit_name,unit_id,country_code,installed_capacity_MW,efficiency,fuel_type,category,is_chp,source_capacity_status,source_efficiency,source_is_chp,source_unit_id,efficiency_1,efficiency_1_source
0,Ledvice B6,OEU-453,CZ,660.0,0.421,lignite,lignite,0,GEM Wiki - Ledvice power station (edited 25.10...,0.421 = 0.357 / 0.847 t CO2/MWh - unit emissio...,own assessment - condensing system power plant,unit id in OPSD EU (OEU-453),0.425,"""up to 42.5%"" (net/gross not stated) - oEnerge..."
1,Tisova Phase,PPM-1258,CZ,276.0,0.330,lignite,lignite,0,"powerplantmatching v0.9.1 (PyPSA, data as of A...","class ""old lignite unit"" - Instrat Foundation,...",own assessment - condensing system power plant,powerplantmatching id (plant-level record),NaN,NaN
2,Prunerov 2 B23,OEU-500,CZ,250.0,0.389,lignite,lignite,0,GEM Wiki - Prunerov power station (edited 17.1...,0.389 = 0.357 / 0.918 t CO2/MWh - unit emissio...,own assessment - condensing system power plant,unit id in OPSD EU (OEU-500),NaN,NaN
3,Prunerov 2 B24,OEU-501,CZ,250.0,0.389,lignite,lignite,0,GEM Wiki - Prunerov power station (edited 17.1...,0.389 = 0.357 / 0.918 t CO2/MWh - unit emissio...,own assessment - condensing system power plant,unit id in OPSD EU (OEU-501),NaN,NaN
4,Prunerov 2 B25,OEU-502,CZ,250.0,0.389,lignite,lignite,0,GEM Wiki - Prunerov power station (edited 17.1...,0.389 = 0.357 / 0.918 t CO2/MWh - unit emissio...,own assessment - condensing system power plant,unit id in OPSD EU (OEU-502),NaN,NaN


In [4]:
power_units.info()

<class 'pandas.DataFrame'>
RangeIndex: 492 entries, 0 to 491
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   unit_name               492 non-null    str    
 1   unit_id                 492 non-null    str    
 2   country_code            492 non-null    str    
 3   installed_capacity_MW   492 non-null    float64
 4   efficiency              492 non-null    float64
 5   fuel_type               492 non-null    str    
 6   category                492 non-null    str    
 7   is_chp                  492 non-null    int64  
 8   source_capacity_status  492 non-null    str    
 9   source_efficiency       492 non-null    str    
 10  source_is_chp           492 non-null    str    
 11  source_unit_id          492 non-null    str    
 12  efficiency_1            125 non-null    float64
 13  efficiency_1_source     125 non-null    str    
dtypes: float64(3), int64(1), str(10)
memory usage: 53.9 K

Sprawdzam, jakie wartości są w kolumnach `fuel_type` i `category`.
`fuel_type` ma czasem kilka typów naraz (np. `gas - CCGT / gas - steam turbine`), a `category` ma jedną krótką nazwę, więc do mapowania biorę `category`.

In [5]:
power_units["fuel_type"].value_counts()

fuel_type
hard coal                                                                                                                       112
gas - CCGT                                                                                                                      100
lignite                                                                                                                          84
gas - steam turbine (gas-fired boiler)                                                                                           57
oil                                                                                                                              44
solid biomass                                                                                                                    38
nuclear (uranium)                                                                                                                18
gas - OCGT                                                        

In [6]:
power_units["category"].value_counts()

category
hard_coal         109
gas_ccgt           97
lignite            82
gas_other          63
oil                43
biomass            36
nuclear            18
gas_ocgt            8
hydro               6
pumped_storage      6
battery             6
chp_gas             5
chp_biomass         4
chp_hard_coal       3
gas                 3
chp_lignite         2
chp_oil             1
Name: count, dtype: int64

In [7]:
power_units["country_code"].value_counts()

country_code
DE    231
PL    115
CZ     67
SE     51
SK     18
LT     10
Name: count, dtype: int64

Bloki DE w rezerwie sieciowej mają w nazwie `[grid reserve]`. Zostawiam je w sumie (ENTSO-E prawdopodobnie też liczy je w mocy zainstalowanej – do sprawdzenia),
ale pokazuję, ile to MW, żeby było wiadomo, skąd może brać się różnica.

In [8]:
rezerwa_sieciowa = power_units[power_units["unit_name"].str.contains("[grid reserve]", regex=False)]
print("Liczba bloków w rezerwie sieciowej:", len(rezerwa_sieciowa))
rezerwa_sieciowa.groupby(["country_code", "category"])["installed_capacity_MW"].sum()

Liczba bloków w rezerwie sieciowej: 28


country_code  category 
DE            gas_ccgt      726.0
              gas_ocgt      147.0
              gas_other     644.0
              hard_coal    7846.0
              oil           857.0
Name: installed_capacity_MW, dtype: float64

## Mapa kategorii na wspólne grupy

Obie strony (baza bloków i ENTSO-E) przypisuję do tych samych grup o polskich nazwach. Najpierw słownik dla kolumny `category`.
Kategorie `chp_...` (elektrociepłownie) idą do grupy swojego paliwa.

In [9]:
category_na_grupe = {
    "hard_coal": "węgiel kamienny",
    "chp_hard_coal": "węgiel kamienny",
    "lignite": "węgiel brunatny",
    "chp_lignite": "węgiel brunatny",
    "gas_ccgt": "gaz",
    "gas_ocgt": "gaz",
    "gas_other": "gaz",
    "gas": "gaz",
    "chp_gas": "gaz",
    "oil": "olej",
    "chp_oil": "olej",
    "nuclear": "jądrowe",
    "biomass": "biomasa",
    "chp_biomass": "biomasa",
    "hydro": "woda (bez ESP)",
    "pumped_storage": "ESP",
    "battery": "bateria",
}

In [10]:
power_units["grupa"] = power_units["category"].map(category_na_grupe)

# sprawdzenie: czy jakaś kategoria nie dostała grupy (powinno być 0 wierszy)
power_units[power_units["grupa"].isna()]

,unit_name,unit_id,country_code,installed_capacity_MW,efficiency,fuel_type,category,is_chp,source_capacity_status,source_efficiency,source_is_chp,source_unit_id,efficiency_1,efficiency_1_source,grupa


Sumuję moc zainstalowaną po kraju i grupie.

In [11]:
moce_power_units = power_units.groupby(["country_code", "grupa"])["installed_capacity_MW"].sum().reset_index()
moce_power_units = moce_power_units.rename(columns={"country_code": "kraj", "installed_capacity_MW": "MW_power_units"})
moce_power_units

,kraj,grupa,MW_power_units
0,CZ,ESP,1175.000
1,CZ,bateria,42.000
2,CZ,biomasa,30.000
3,CZ,gaz,1818.600
4,CZ,jądrowe,4164.000
5,CZ,woda (bez ESP),674.600
6,CZ,węgiel brunatny,6737.500
7,CZ,węgiel kamienny,667.200
8,DE,ESP,7230.500
9,DE,bateria,21951.000


In [12]:
# suma per kraj, dla kontroli
moce_power_units.groupby("kraj")["MW_power_units"].sum()

kraj
CZ     15308.900
DE    112863.328
LT      2635.000
PL     34500.200
SE     30014.100
SK      6825.200
Name: MW_power_units, dtype: float64

# Pobranie danych z ENTSO-E

Klucz API wczytuję z pliku `.env` (w głównym folderze projektu, czyli `../.env`). Funkcja `load_dotenv` wczytuje plik,
a `os.getenv` zwraca wartość zmiennej. Klucza nie wypisuję, sprawdzam tylko, czy się wczytał.

In [ ]:
load_dotenv("../.env")
klucz_api = os.getenv("ENTSOE_API_KEY")
print("Klucz wczytany:", klucz_api is not None)

In [ ]:
client = EntsoePandasClient(api_key=klucz_api)

Kody krajów w `entsoe-py`. Dla DE używam kodu kraju `"DE"` (Niemcy). Jeśli zapytanie zwróci błąd lub puste dane, można spróbować strefy `"DE_LU"`.
SE pobieram dla całego kraju (`"SE"`), bo baza bloków ma kod kraju, a nie strefy SE1–SE4.

Moc zainstalowana w ENTSO-E jest podawana raz na rok, więc biorę zakres od 1.01.2025 do 31.12.2025.

In [ ]:
kraje_entsoe = {
    "PL": "PL",
    "DE": "DE",
    "CZ": "CZ",
    "SK": "SK",
    "LT": "LT",
    "SE": "SE",
}

poczatek = pd.Timestamp("2025-01-01", tz="Europe/Brussels")
koniec = pd.Timestamp("2025-12-31", tz="Europe/Brussels")

Pobieram moc zainstalowaną per typ produkcji dla każdego kraju i zapisuję w słowniku `{kraj: df}`.
Każde zapytanie trwa kilka sekund.

In [ ]:
moce_entsoe = {}
for kraj, kod_entsoe in kraje_entsoe.items():
    print("Pobieram:", kraj)
    moce_entsoe[kraj] = client.query_installed_generation_capacity(kod_entsoe, start=poczatek, end=koniec)

Sprawdzenie: jak wygląda wynik dla jednego kraju. Wiersz = rok, kolumny = typy produkcji ENTSO-E, wartości w MW.

In [ ]:
moce_entsoe["PL"]

In [ ]:
# sprawdzenie: ile wierszy (lat) przyszło dla każdego kraju (powinien być 1 wiersz = 2025)
for kraj, moce in moce_entsoe.items():
    print(kraj, moce.shape, list(moce.index))

Zamieniam słownik na jedną długą tabelę: kolumny `kraj`, `typ_entsoe`, `MW_entsoe`. Biorę pierwszy wiersz (`.iloc[0]`), czyli rok 2025.

In [ ]:
wiersze = []
for kraj, moce in moce_entsoe.items():
    for typ_entsoe in moce.columns:
        wiersze.append({"kraj": kraj, "typ_entsoe": typ_entsoe, "MW_entsoe": moce[typ_entsoe].iloc[0]})

moce_entsoe_tabela = pd.DataFrame(wiersze)
moce_entsoe_tabela

Zapisuję pobrane dane do CSV, żeby następnym razem nie pytać API (folder `data/` nie trafia do gita).

In [ ]:
moce_entsoe_tabela.to_csv("../data/moce_zainstalowane_entsoe_2025.csv", index=False)

## Mapa typów produkcji ENTSO-E na wspólne grupy

- `Fossil Coal-derived gas` (CZ Vřesová) liczę jako gaz.
- Wiatr, PV, odpady, geotermia, `Other` i `Other renewable` pomijam, bo nie ma ich w bazie bloków.
- Uwaga: dla SE kategoria `Other` to biomasa + olej + gaz, więc w SE te grupy w ENTSO-E mogą wyjść za małe.

In [ ]:
entsoe_na_grupe = {
    "Fossil Hard coal": "węgiel kamienny",
    "Fossil Brown coal/Lignite": "węgiel brunatny",
    "Fossil Gas": "gaz",
    "Fossil Coal-derived gas": "gaz",
    "Fossil Oil": "olej",
    "Nuclear": "jądrowe",
    "Biomass": "biomasa",
    "Hydro Run-of-river and pondage": "woda (bez ESP)",
    "Hydro Water Reservoir": "woda (bez ESP)",
    "Hydro Pumped Storage": "ESP",
    "Energy storage": "bateria",
}

In [ ]:
moce_entsoe_tabela["grupa"] = moce_entsoe_tabela["typ_entsoe"].map(entsoe_na_grupe)

# sprawdzenie: które typy pomijam (brak grupy)
pominiete = moce_entsoe_tabela[moce_entsoe_tabela["grupa"].isna()]
pominiete

In [ ]:
moce_entsoe_grupy = moce_entsoe_tabela.groupby(["kraj", "grupa"])["MW_entsoe"].sum().reset_index()
moce_entsoe_grupy

## (Opcjonalnie) Moce per jednostka z ENTSO-E

ENTSO-E publikuje też moc per jednostka (bloki ≥ 100 MW). Bloki mają tam kody EIC, a w bazie bloków są inne identyfikatory (`OEU-...`, `PPM-...`),
więc nie da się ich prosto połączyć po ID. Tu tylko pobieram listę dla jednego kraju i ją oglądam – dopasowanie po nazwach to osobne zadanie.

In [ ]:
jednostki_pl = client.query_installed_generation_capacity_per_unit("PL", start=poczatek, end=koniec)
jednostki_pl

In [ ]:
# moc jest zapisana jako tekst, więc zamieniam ją na liczbę
jednostki_pl["Installed Capacity [MW]"] = pd.to_numeric(jednostki_pl["Installed Capacity [MW]"])
jednostki_pl.groupby("Production Type")["Installed Capacity [MW]"].sum()

# Porównanie

Łączę obie tabele po kraju i grupie (`how="outer"`, żeby zostały też grupy, które są tylko po jednej stronie). Brak wartości = 0 MW.

In [ ]:
porownanie = pd.merge(moce_power_units, moce_entsoe_grupy, on=["kraj", "grupa"], how="outer")
porownanie["MW_power_units"] = porownanie["MW_power_units"].fillna(0)
porownanie["MW_entsoe"] = porownanie["MW_entsoe"].fillna(0)
porownanie

Różnica = baza bloków − ENTSO-E. Dodatnia różnica znaczy, że w bazie bloków jest więcej MW niż w ENTSO-E.
Procent liczę względem ENTSO-E, tylko tam, gdzie ENTSO-E ma więcej niż 0 MW (żeby nie dzielić przez zero).

In [ ]:
porownanie["roznica_MW"] = porownanie["MW_power_units"] - porownanie["MW_entsoe"]

porownanie["roznica_%"] = np.nan
czy_entsoe_powyzej_zera = porownanie["MW_entsoe"] > 0
porownanie.loc[czy_entsoe_powyzej_zera, "roznica_%"] = porownanie["roznica_MW"] / porownanie["MW_entsoe"] * 100

porownanie = porownanie.round(1)
porownanie

Duże rozbieżności (więcej niż ±20% albo ±500 MW):

In [ ]:
porownanie[(porownanie["roznica_%"].abs() > 20) | (porownanie["roznica_MW"].abs() > 500)]

Tabela kraj × grupa z różnicą w MW (`pivot`).

In [ ]:
porownanie.pivot(index="grupa", columns="kraj", values="roznica_MW")

Sumy per kraj (tylko grupy z mapy, czyli bez wiatru i PV).

In [ ]:
sumy_kraje = porownanie.groupby("kraj")[["MW_power_units", "MW_entsoe", "roznica_MW"]].sum()
sumy_kraje

In [ ]:
porownanie.to_csv("../data/porownanie_mocy_power_units_entsoe_2025.csv", index=False)

# Wykresy

Dla każdego kraju wykres słupkowy: dwa słupki obok siebie dla każdej grupy paliw (baza bloków i ENTSO-E).
`np.arange` daje pozycje 0, 1, 2, …, a słupki przesuwam o pół szerokości w lewo i w prawo.

In [ ]:
szerokosc = 0.4

for kraj in kraje_entsoe:
    dane_kraju = porownanie[porownanie["kraj"] == kraj]
    pozycje = np.arange(len(dane_kraju))

    plt.figure(figsize=(18, 6))
    plt.bar(pozycje - szerokosc / 2, dane_kraju["MW_power_units"], width=szerokosc, label="baza bloków (power_units)")
    plt.bar(pozycje + szerokosc / 2, dane_kraju["MW_entsoe"], width=szerokosc, label="ENTSO-E 2025")
    plt.xticks(pozycje, dane_kraju["grupa"])
    plt.title("Moc zainstalowana – " + kraj)
    plt.ylabel("MW")
    plt.legend()
    plt.show()

Różnica w % dla wszystkich krajów na jednym wykresie (słupki dla każdej grupy, kolory = kraje).

In [ ]:
grupy = list(category_na_grupe.values())
grupy_bez_powtorzen = []
for grupa in grupy:
    if grupa not in grupy_bez_powtorzen:
        grupy_bez_powtorzen.append(grupa)

pozycje = np.arange(len(grupy_bez_powtorzen))
szerokosc_kraju = 0.13

plt.figure(figsize=(18, 6))
numer_kraju = 0
for kraj in kraje_entsoe:
    dane_kraju = porownanie[porownanie["kraj"] == kraj]
    dane_kraju = dane_kraju.set_index("grupa").reindex(grupy_bez_powtorzen)
    plt.bar(pozycje + numer_kraju * szerokosc_kraju, dane_kraju["roznica_%"], width=szerokosc_kraju, label=kraj)
    numer_kraju = numer_kraju + 1

plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(pozycje + 2.5 * szerokosc_kraju, grupy_bez_powtorzen)
plt.title("Różnica mocy: baza bloków vs ENTSO-E 2025")
plt.ylabel("różnica [%]")
plt.legend()
plt.show()